# Parecer prévio por IA: avaliação, portão de qualidade, registro e endpoint

**Problema de negócio.** O regulador gasta a maior parte do tempo lendo orçamento, laudo, apólice e dados do
clima para chegar a uma recomendação que, na maioria dos casos, é previsível. O **parecer prévio** faz essa
leitura e entrega ao regulador:
- a recomendação (`pagar_integral`, `pagar_parcial`, `recusar` ou `investigar`);
- o valor sugerido;
- os motivos e as cláusulas.

A decisão continua com o regulador: a IA nunca recusa sozinha.

**Como funciona (D-022).**
- O Claude Sonnet aplica as regras às evidências (`silver.entrada_parecer` + score do `modelo_suspeita@champion`)
  e escolhe a recomendação e a **base** do valor.
- O código valida o JSON (com uma nova tentativa) e **calcula o valor**: franquia, LMI e itens excluídos.

**O que este notebook faz.**
1. Dataset estratificado dos lotes de validação do evento, com o gabarito como expectativa, salvo como MLflow
   Evaluation Dataset no UC.
2. Prompts v1 (básico) e v2 (regras explícitas e exemplos) versionados no UC (tabela Delta, porque o
   Prompt Registry não está habilitado no workspace; D-022).
3. `mlflow.genai.evaluate()` dos dois prompts, com métricas de código e juízes LLM (`Correctness` e `Guidelines`).
4. Portão de qualidade: o alias `producao` do prompt só muda se `recomendacao_correta` não piorar.
5. Registro de `ml.parecer_sinistro` (*models from code*, com o prompt de produção congelado) e publicação do
   endpoint com AI Gateway (inference table e usage tracking) e tracing.

**Leitura honesta.**
- O modelo não aceita `temperature`, então há variação entre execuções.
- O dataset é pequeno (até 25 casos por padrão).
- Diferenças de poucos pontos entre v1 e v2 estão dentro do ruído.

In [ ]:
import json
import os
import tempfile
from importlib.metadata import version as versao_pacote
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from databricks.sdk import WorkspaceClient
from mlflow import MlflowClient
from mlflow.genai.scorers import Correctness, Guidelines, scorer
from mlflow.models.resources import DatabricksServingEndpoint
from pyspark.sql import functions as F

import parecer
from parecer import avaliacao as av
from parecer.endpoint import api_do_workspace, config_endpoint, publicar_endpoint
from parecer.esquema import RECOMENDACOES
from parecer.gerar import chamar_llm_databricks, gerar_parecer, juntar_score
from parecer.modelo_pyfunc import assinatura_parecer
from parecer.prompts import PROMPTS
from regras.features import COLUNAS_ENTRADA, preparar_entrada

PARAMETROS = {"catalogo": "", "schema_silver": "", "schema_ml": "", "experimento": "", "endpoint_llm": "",
              "endpoint_parecer": "", "lotes_validacao": "1", "max_por_padrao": "25", "publicar": "sim"}
for nome, padrao in PARAMETROS.items():
    dbutils.widgets.text(nome, padrao)  # noqa: F821 (dbutils é injetado pelo Databricks)
P = {nome: dbutils.widgets.get(nome) for nome in PARAMETROS}  # noqa: F821
assert all(P.values()), f"parâmetros do job ausentes: {[k for k, v in P.items() if not v]}"
SILVER, ML = f"{P['catalogo']}.{P['schema_silver']}", f"{P['catalogo']}.{P['schema_ml']}"
LOTES = sorted({int(x) for x in P["lotes_validacao"].split(",") if x.strip()})
# publicar = "nao": só avalia (ex.: comparar outro LLM no mesmo dataset), sem mexer no alias, no modelo nem no endpoint
PUBLICAR = P["publicar"].strip().lower() == "sim"
SUFIXO_RUN = "" if PUBLICAR else f"_{P['endpoint_llm'].removeprefix('databricks-')}"
NOME_PROMPT, NOME_MODELO = f"{ML}.prompt_parecer", f"{ML}.parecer_sinistro"
NOME_DATASET = f"{ML}.avaliacao_parecer_dataset"
GABARITO = f"/Volumes/{P['catalogo']}/{P['schema_ml']}/restrito/gabarito"
VERSOES = {p: versao_pacote(p) for p in ("mlflow", "databricks-sdk", "pandas", "numpy")}
os.environ.setdefault("MLFLOW_GENAI_EVAL_MAX_WORKERS", "8")  # paralelismo moderado no FMAPI (pay-per-token)
mlflow.set_registry_uri("databricks-uc")
EXPERIMENTO_ID = mlflow.set_experiment(P["experimento"]).experiment_id
print(f"prompt: {NOME_PROMPT} | modelo: {NOME_MODELO} | dataset: {NOME_DATASET} | LLM: {P['endpoint_llm']} | "
      f"endpoint: {P['endpoint_parecer']} | lotes: {LOTES}")
print("versões:", VERSOES)

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


prompt: fe_bar_sinistros_catalog.ml.prompt_parecer | modelo: fe_bar_sinistros_catalog.ml.parecer_sinistro | dataset: fe_bar_sinistros_catalog.ml.avaliacao_parecer_dataset | LLM: databricks-claude-sonnet-5-5 | endpoint: sinistros-clima-parecer-dev | lotes: [1]
versões: {'mlflow': '3.18.0', 'databricks-sdk': '0.169.0', 'pandas': '2.2.3', 'numpy': '2.1.3'}


## 1. Dataset de avaliação

- **Casos:** sinistros dos lotes de validação, com a entrada do parecer e o score do `modelo_suspeita@champion`.
- **Amostra:** estratificada, com até `max_por_padrao` casos por padrão.
- **Expectativas, só do gabarito restrito:** recomendação esperada, valor devido, fatos esperados (juiz
  `Correctness`) e a cláusula que um parecer correto cita. O gabarito nunca entra na entrada do parecer.

In [ ]:
lote = (spark.table(f"{SILVER}.avisos")  # noqa: F821
        .select("id_aviso", F.regexp_extract("_arquivo", r"avisos_lote_([0-9]+)", 1).cast("int").alias("lote")))
gab = (spark.read.csv(f"{GABARITO}/gabarito_evento_2026.csv", header=True)  # noqa: F821
       .select("id_aviso", "padrao", "decisao_esperada", F.col("valor_devido").cast("double").alias("valor_devido")))
casos = (spark.table(f"{SILVER}.entrada_parecer")  # noqa: F821
         .join(spark.table(f"{SILVER}.sinistros_enriquecidos").select("id_aviso", "cobertura_necessaria", *COLUNAS_ENTRADA),  # noqa: F821
               "id_aviso")
         .join(lote, "id_aviso").where(F.col("lote").isin(LOTES))
         .join(gab, "id_aviso").orderBy("id_aviso").toPandas())
assert not casos.empty, f"nenhum sinistro dos lotes {LOTES} no pipeline"
suspeita = mlflow.pyfunc.load_model(f"models:/{ML}.modelo_suspeita@champion")
saida_suspeita = suspeita.predict(preparar_entrada(casos))
casos["score_suspeita"] = saida_suspeita["score_suspeita"].to_numpy()
casos["fatores_suspeita"] = saida_suspeita["fatores_suspeita"].to_numpy()
amostra = av.amostra_estratificada(casos, "padrao", int(P["max_por_padrao"]), seed=42)
registros = [{"inputs": {"entrada": juntar_score(json.loads(r.entrada_json), r.score_suspeita, r.fatores_suspeita)},
              "expectations": av.expectativas(r.padrao, r.decisao_esperada, r.valor_devido, r.cobertura_necessaria)}
             for r in amostra.itertuples()]

if not PUBLICAR:  # comparação de LLM: usa os mesmos casos, sem recriar o dataset de produção
    dataset = registros
try:  # recriado a cada execução publicada: o dataset reflete os lotes de validação atuais
    mlflow.genai.datasets.get_dataset(name=NOME_DATASET)
    existe = True
except Exception as e:  # noqa: BLE001 (o código do erro de "não existe" varia; qualquer outro erro reaparece no create)
    existe, motivo = False, f"{type(e).__name__}: {str(e)[:120]}"
if existe and PUBLICAR:
    mlflow.genai.datasets.delete_dataset(name=NOME_DATASET)
if PUBLICAR:
    # no Databricks, create_dataset não aceita tags (a governança é do UC): a descrição vai como COMMENT da tabela
    dataset = mlflow.genai.datasets.create_dataset(name=NOME_DATASET, experiment_id=[EXPERIMENTO_ID])
    dataset.merge_records(registros)
    spark.sql(f"COMMENT ON TABLE {NOME_DATASET} IS 'Avaliação do parecer prévio: {len(registros)} casos estratificados "  # noqa: F821
              f"dos lotes de validação {P['lotes_validacao']} do evento, com expectativas do gabarito (spec 6.4).'")
print(f"{len(registros)} casos de {len(casos)} sinistros dos lotes {LOTES} → {NOME_DATASET}")
display(amostra.groupby("padrao").size().rename("casos").reset_index())  # noqa: F821

Unexpected status resolving file endpoints (403, request ID b380187e-41c6-4c38-92c6-b42213ecaf26), using the default endpoints.


[imagem: ver os artefatos do run no MLflow]


143 casos de 800 sinistros dos lotes [1] → fe_bar_sinistros_catalog.ml.avaliacao_parecer_dataset


| padrao | casos |
|---|---|
| apolice_recente | 13 |
| dano_preexistente | 9 |
| legitimo_complexo | 25 |
| legitimo_simples | 25 |
| orcamento_inflado | 25 |
| prestador_concentrado | 5 |
| sem_cobertura | 16 |
| sem_evento_climatico | 25 |


## 2. Prompts versionados no UC

O Prompt Registry do MLflow não está habilitado neste workspace (`FEATURE_DISABLED`; D-022). Os prompts ficam
numa tabela Delta do UC (`ml.prompt_parecer`), com as mesmas operações:
- versão nova só quando o texto muda (sha256);
- alias de produção;
- métrica de qualidade por versão;
- o histórico Delta da tabela é a trilha de auditoria.

In [ ]:
ESQUEMA_PROMPTS = ("versao BIGINT, rotulo STRING, template STRING, sha256 STRING, mensagem STRING, criado_em STRING, "
                   "recomendacao_correta DOUBLE, producao BOOLEAN")


def _ler_prompts() -> pd.DataFrame:
    if not spark.catalog.tableExists(NOME_PROMPT):  # noqa: F821
        return av.tabela_prompts_vazia()
    return spark.table(NOME_PROMPT).toPandas()[av.COLUNAS_PROMPTS]  # noqa: F821


def _gravar_prompts(tabela: pd.DataFrame) -> None:
    (spark.createDataFrame(tabela[av.COLUNAS_PROMPTS], ESQUEMA_PROMPTS)  # noqa: F821
     .write.mode("overwrite").saveAsTable(NOME_PROMPT))
    spark.sql(f"COMMENT ON TABLE {NOME_PROMPT} IS 'Versões do prompt do parecer prévio (uma linha por versão; producao = "  # noqa: F821
              "alias de produção). Histórico Delta = auditoria (D-022).'")


TABELA_PROMPTS = _ler_prompts()
VERSOES_PROMPT = {}
agora = pd.Timestamp.now(tz="UTC").isoformat()
for chave, texto in PROMPTS.items():
    mensagem = f"Parecer prévio {chave}: " + ("básico" if chave == "v1" else "regras explícitas, ordem e exemplos")
    TABELA_PROMPTS, VERSOES_PROMPT[chave], nova = av.registrar_versao(TABELA_PROMPTS, texto, chave, mensagem, agora)
    print(f"{chave}: versão {VERSOES_PROMPT[chave]} ({'nova' if nova else 'reaproveitada'})")
if PUBLICAR:
    _gravar_prompts(TABELA_PROMPTS)

v1: versão 1 (nova)
v2: versão 2 (nova)


## 3. Avaliação dos dois prompts (`mlflow.genai.evaluate`)

**Métricas de código:**
- `recomendacao_correta`: recomendação igual à do gabarito;
- `json_valido`: o parecer passou na validação;
- `padrao_identificado`: os motivos citam o fato que caracteriza o padrão;
- `cita_clausula`: cita a cláusula esperada;
- `valor_dentro_tolerancia`: ±10% do devido, só nos casos com pagamento esperado, calculado fora dos scorers para
  não misturar tipos.

**Juízes LLM:**
- `Correctness`: os fatos esperados estão no parecer;
- `Guidelines`: português formal, não recusar sem evidência objetiva, citar cobertura ou evidência, tom
  respeitoso.

Cada caso gera um trace.

In [ ]:
@scorer(name="recomendacao_correta")
def s_recomendacao(outputs, expectations):
    return av.recomendacao_correta(outputs, expectations)


@scorer(name="json_valido")
def s_json(outputs):
    return av.json_valido(outputs)


@scorer(name="padrao_identificado")
def s_padrao(outputs, expectations):
    return av.padrao_identificado(outputs, expectations)


@scorer(name="cita_clausula")
def s_clausula(outputs, expectations):
    return av.cita_clausula(outputs, expectations)


DIRETRIZES = Guidelines(name="diretrizes_parecer", guidelines=[
    "A resposta está em português formal.",
    "Não recomenda recusar sem citar a ausência de cobertura ou outra evidência objetiva dos dados do sinistro.",
    "Os motivos citam a cobertura, a evidência climática ou os documentos que fundamentam a recomendação.",
    "O tom é respeitoso com o segurado e não faz acusações de fraude; descreve fatos.",
])
SCORERS = [s_recomendacao, s_json, s_padrao, s_clausula, Correctness(), DIRETRIZES]
chamar = chamar_llm_databricks(P["endpoint_llm"])


def _tabela(resultado) -> pd.DataFrame:
    df = resultado.result_df
    resposta = df["response"].apply(lambda r: r if isinstance(r, dict) else {})
    t = pd.DataFrame({
        "id_aviso": df["request"].apply(lambda r: r["entrada"]["aviso"].get("id_aviso")),
        "padrao": df["padrao/value"], "esperado": df["recomendacao_esperada/value"],
        "obtido": resposta.apply(lambda r: r.get("recomendacao")),
        "valor_devido": df["valor_devido/value"].astype(float),
        "valor_sugerido": resposta.apply(lambda r: r.get("valor_sugerido")).astype(float),
        "status": resposta.apply(lambda r: r.get("status_parecer")),
        "dias_desde_contratacao": df["request"].apply(lambda r: (r["entrada"].get("apolice") or {}).get("dias_desde_contratacao")),
        "motivos": resposta.apply(lambda r: " | ".join(r.get("motivos") or [])[:300]),
        "trace_id": df["trace_id"]})
    t["correto"] = t["esperado"] == t["obtido"]
    return t


def _figura_confusao(t: pd.DataFrame, titulo: str):
    m = pd.crosstab(t["esperado"], t["obtido"]).reindex(index=RECOMENDACOES, columns=RECOMENDACOES, fill_value=0)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.imshow(m.to_numpy(), cmap="Blues")
    for (i, j), v in np.ndenumerate(m.to_numpy()):
        ax.text(j, i, str(v), ha="center", va="center")
    ax.set(xticks=range(4), yticks=range(4), xticklabels=RECOMENDACOES, yticklabels=RECOMENDACOES,
           xlabel="recomendação do parecer", ylabel="recomendação esperada (gabarito)", title=titulo)
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    fig.tight_layout()
    return fig, m


RESULTADOS, TABELAS, RUNS = {}, {}, {}
for chave, versao in VERSOES_PROMPT.items():
    template = PROMPTS[chave]

    def predict_fn(entrada, _template=template, _versao=str(versao)):
        return gerar_parecer(entrada, _template, chamar, _versao)

    with mlflow.start_run(run_name=f"prompt_{chave}_versao_{versao}{SUFIXO_RUN}") as run:
        RUNS[chave] = run.info.run_id
        mlflow.log_params({"prompt": NOME_PROMPT, "prompt_versao": versao, "prompt_rotulo": chave,
                           "llm": P["endpoint_llm"], "casos": len(registros), "dataset": NOME_DATASET,
                           "lotes_validacao": P["lotes_validacao"], **{f"versao_{k}": v for k, v in VERSOES.items()}})
        mlflow.log_text(template, f"prompt_{chave}_v{versao}.txt")  # o texto avaliado fica junto do run
        resultado = mlflow.genai.evaluate(data=dataset, predict_fn=predict_fn, scorers=SCORERS)
        t = _tabela(resultado)
        pagamentos = t[t["esperado"].str.startswith("pagar")]
        extras = {"valor_dentro_tolerancia_pagamentos": float(np.mean([
                      av.valor_dentro_tolerancia(v, d) for v, d in zip(pagamentos["valor_sugerido"], pagamentos["valor_devido"])]))
                  if len(pagamentos) else float("nan"),
                  "casos_pagamento_esperado": float(len(pagamentos))}
        mlflow.log_metrics(extras)
        por_padrao = (t.groupby("padrao").agg(casos=("correto", "size"), acuracia=("correto", "mean"))
                      .round(3).reset_index())
        erros = t[~t["correto"]].drop(columns=["correto"])
        fig, matriz = _figura_confusao(t, f"Prompt {chave} (versão {versao})")
        mlflow.log_figure(fig, "matriz_confusao_recomendacoes.png")
        mlflow.log_table(por_padrao, "acuracia_por_padrao.json")
        mlflow.log_table(erros, "erros.json")
        mlflow.log_table(t.drop(columns=["motivos"]), "resultados.json")
        plt.close("all")
    RESULTADOS[chave] = {**{k: float(v) for k, v in resultado.metrics.items()}, **extras}
    TABELAS[chave] = {"por_padrao": por_padrao, "erros": erros, "matriz": matriz, "tabela": t}
    if PUBLICAR:  # a métrica gravada no prompt é sempre a do LLM de produção
        TABELA_PROMPTS = av.gravar_metrica(TABELA_PROMPTS, versao, RESULTADOS[chave]["recomendacao_correta/mean"])

    print(f"prompt {chave} (v{versao}): " + ", ".join(f"{k}={v:.3f}" for k, v in sorted(RESULTADOS[chave].items())))

comparacao = pd.DataFrame(RESULTADOS).round(3).rename_axis("metrica").reset_index()
display(comparacao)  # noqa: F821
display(TABELAS["v1"]["por_padrao"].merge(TABELAS["v2"]["por_padrao"], on=["padrao", "casos"], suffixes=("_v1", "_v2")))  # noqa: F821

Unexpected status resolving file endpoints (403, request ID da7eaff4-1cf1-44e6-9751-cc9d4971a238), using the default endpoints.
2026/11/21 03:35:10 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


[imagem: ver os artefatos do run no MLflow]


[imagem: ver os artefatos do run no MLflow]


prompt v1 (v1): casos_pagamento_esperado=84.000, cita_clausula/mean=0.951, correctness/mean=0.664, diretrizes_parecer/mean=0.685, json_valido/mean=0.986, padrao_identificado/mean=0.895, recomendacao_correta/mean=0.664, valor_dentro_tolerancia_pagamentos=0.583


Unexpected status resolving file endpoints (403, request ID e635d382-5d06-4859-b6b6-3ce850081d6a), using the default endpoints.
2026/11/21 03:39:09 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


[imagem: ver os artefatos do run no MLflow]


[imagem: ver os artefatos do run no MLflow]


prompt v2 (v2): casos_pagamento_esperado=84.000, cita_clausula/mean=0.951, correctness/mean=0.930, diretrizes_parecer/mean=0.923, json_valido/mean=1.000, padrao_identificado/mean=0.993, recomendacao_correta/mean=0.951, valor_dentro_tolerancia_pagamentos=1.000


| metrica | v1 | v2 |
|---|---|---|
| recomendacao_correta/mean | 0.664 | 0.951 |
| json_valido/mean | 0.986 | 1.0 |
| cita_clausula/mean | 0.951 | 0.951 |
| padrao_identificado/mean | 0.895 | 0.993 |
| diretrizes_parecer/mean | 0.685 | 0.923 |
| correctness/mean | 0.664 | 0.93 |
| valor_dentro_tolerancia_pagamentos | 0.583 | 1.0 |
| casos_pagamento_esperado | 84.0 | 84.0 |


| padrao | casos | acuracia_v1 | acuracia_v2 |
|---|---|---|---|
| apolice_recente | 13 | 1.0 | 1.0 |
| dano_preexistente | 9 | 0.778 | 1.0 |
| legitimo_complexo | 25 | 0.24 | 0.96 |
| legitimo_simples | 25 | 0.4 | 1.0 |
| orcamento_inflado | 25 | 0.72 | 1.0 |
| prestador_concentrado | 5 | 0.0 | 0.0 |
| sem_cobertura | 16 | 1.0 | 1.0 |
| sem_evento_climatico | 25 | 1.0 | 0.96 |


### Erros do prompt v2

Os casos em que o parecer v2 diverge do gabarito, com os motivos. Apólices com menos de 30 dias vão para
`investigar` pela cláusula CG-2.4 (análise de contratação recente), mesmo quando o gabarito diz que o sinistro é
legítimo. Essa divergência é de política, não de leitura.

In [ ]:
erros_v2 = TABELAS["v2"]["erros"]
display(erros_v2[["id_aviso", "padrao", "esperado", "obtido", "dias_desde_contratacao", "motivos"]])  # noqa: F821
print("Matriz de confusão v2 (linhas: esperado; colunas: obtido):")
print(TABELAS["v2"]["matriz"].to_string())
politica = erros_v2[(erros_v2["obtido"] == "investigar") & (pd.to_numeric(erros_v2["dias_desde_contratacao"]) < 30)]
print(f"{len(erros_v2)} divergências no v2; {len(politica)} delas são apólices com menos de 30 dias (CG-2.4).")

| id_aviso | padrao | esperado | obtido | dias_desde_contratacao | motivos |
|---|---|---|---|---|---|
| AV-2026-000748 | prestador_concentrado | investigar | pagar_integral | 44.0 | Causa declarada (vendaval) enquadrada na cobertura de vendaval e granizo contratada, com cobertura válida na apólice AP-0036055 | Evento de 28/08/2026 confirmado por registro da Defesa Civil em Florianópolis (evidência forte), apesar de não haver estação meteorológica no raio | Laudo de vistoria de  |
| AV-2026-000301 | prestador_concentrado | investigar | pagar_integral | 94.0 | Cobertura de vendaval e granizo contratada e válida para a causa declarada (granizo), com LMI de R$ 42.000,00 e franquia de R$ 1.000,00 | Evento de 28/08/2026 confirmado por registro da Defesa Civil em Florianópolis (evidência forte); a ausência de estação meteorológica no raio não afasta a comprova |
| AV-2026-000714 | prestador_concentrado | investigar | pagar_integral | 132.0 | Causa declarada (vendaval) corresponde à cobertura

Matriz de confusão v2 (linhas: esperado; colunas: obtido):
obtido          pagar_integral  pagar_parcial  recusar  investigar
esperado                                                          
pagar_integral              48              0        0           1
pagar_parcial                0             35        0           0
recusar                      0              0       16           0
investigar                   6              0        0          37
7 divergências no v2; 1 delas são apólices com menos de 30 dias (CG-2.4).


## 4. Portão de qualidade do prompt (spec 6.4)

O candidato é a versão com a maior `recomendacao_correta`; no empate, o v2, mais explícito. O alias `producao`
só vai para o candidato se a métrica dele for maior ou igual à da versão de produção atual. Se a versão atual foi
avaliada agora, vale a métrica desta execução; senão, a que ficou gravada na tag da versão.

In [ ]:
if not PUBLICAR:
    resultado = {"dataset": NOME_DATASET, "casos": len(registros), "llm": P["endpoint_llm"], "versoes_prompt": VERSOES_PROMPT,
                 "runs": RUNS, "publicar": False,
                 "metricas": {c: {k: round(v, 4) for k, v in m.items()} for c, m in RESULTADOS.items()}}
    print(json.dumps(resultado, ensure_ascii=False, indent=2))
    dbutils.notebook.exit(json.dumps(resultado, ensure_ascii=False))  # noqa: F821

metrica = {c: RESULTADOS[c]["recomendacao_correta/mean"] for c in VERSOES_PROMPT}
candidato = max(VERSOES_PROMPT, key=lambda c: (metrica[c], c == "v2"))
atual = av.versao_producao(TABELA_PROMPTS)
avaliadas = {v: c for c, v in VERSOES_PROMPT.items()}
if atual is None:
    metrica_atual = None
elif int(atual["versao"]) in avaliadas:
    metrica_atual = metrica[avaliadas[int(atual["versao"])]]
else:
    metrica_atual = float(atual["recomendacao_correta"])
PROMOVER = metrica_atual is None or not metrica_atual == metrica_atual or metrica[candidato] >= metrica_atual
if PROMOVER:
    TABELA_PROMPTS = av.definir_producao(TABELA_PROMPTS, VERSOES_PROMPT[candidato])
_gravar_prompts(TABELA_PROMPTS)
PRODUCAO = av.versao_producao(TABELA_PROMPTS)
print(f"candidato {candidato} (v{VERSOES_PROMPT[candidato]}, {metrica[candidato]:.3f}); produção anterior: "
      f"{None if atual is None else atual['versao']} ({metrica_atual}); promovido: {PROMOVER}; "
      f"produção agora: v{PRODUCAO['versao']}")

candidato v2 (v2, 0.951); produção anterior: None (None); promovido: True; produção agora: v2


## 5. Registro do modelo `ml.parecer_sinistro` e publicação do endpoint

- O modelo leva o prompt de produção **congelado**.
- Declara o LLM em `resources`: o endpoint chama o Foundation Model API com passthrough de credenciais.
- O endpoint é criado ou atualizado com scale-to-zero, AI Gateway (inference table e usage tracking) e tracing
  no experimento do parecer.
- O teste do endpoint publicado fica em `scripts/testar_endpoint_parecer.py`.

In [ ]:
pasta = Path(tempfile.mkdtemp())
(pasta / "config.json").write_text(json.dumps({
    "endpoint_llm": P["endpoint_llm"], "nome_prompt": NOME_PROMPT, "versao_prompt": str(PRODUCAO['versao']),
    "template": PRODUCAO['template']}, ensure_ascii=False), encoding="utf-8")
exemplo = pd.DataFrame({"entrada_json": [amostra.iloc[0].entrada_json], "score_suspeita": [float(amostra.iloc[0].score_suspeita)],
                        "fatores_suspeita": [amostra.iloc[0].fatores_suspeita]})
with mlflow.start_run(run_name=f"registro_parecer_prompt_v{PRODUCAO['versao']}") as run:
    RUN_REGISTRO = run.info.run_id
    info = mlflow.pyfunc.log_model(
        name="parecer", python_model=str(Path(parecer.__file__).with_name("modelo_pyfunc.py")),
        artifacts={"config": str(pasta / "config.json")}, code_paths=[str(Path(parecer.__file__).parent)],
        signature=assinatura_parecer(), input_example=exemplo,
        pip_requirements=[f"mlflow=={VERSOES['mlflow']}", f"databricks-sdk=={VERSOES['databricks-sdk']}",
                          f"pandas=={VERSOES['pandas']}", f"numpy=={VERSOES['numpy']}"],
        resources=[DatabricksServingEndpoint(endpoint_name=P["endpoint_llm"])],
        registered_model_name=NOME_MODELO)
VERSAO_MODELO = str(info.registered_model_version)
cliente = MlflowClient(registry_uri="databricks-uc")
cliente.set_registered_model_alias(NOME_MODELO, "producao", VERSAO_MODELO)
for chave, valor in {"prompt": NOME_PROMPT, "versao_prompt": str(PRODUCAO['versao']), "llm": P["endpoint_llm"],
                     "recomendacao_correta": f"{metrica[avaliadas.get(int(PRODUCAO['versao']), candidato)]:.4f}",
                     "dataset": NOME_DATASET}.items():
    cliente.set_model_version_tag(NOME_MODELO, VERSAO_MODELO, chave, valor)
cliente.update_registered_model(NOME_MODELO, description=(
    "Parecer prévio de sinistros patrimoniais (Claude Sonnet via Foundation Model API). Entrada: entrada_json de "
    "silver.entrada_parecer + score e fatores do modelo de suspeita. Saída: recomendação, valor sugerido (calculado em "
    "código), motivos, cláusulas, confiança, status e trace. Alias producao = versão servida."))
saida_registrado = mlflow.pyfunc.load_model(f"models:/{NOME_MODELO}/{VERSAO_MODELO}").predict(exemplo)
print(f"registrado: {NOME_MODELO} versão {VERSAO_MODELO} com o prompt v{PRODUCAO['versao']}; teste: "
      f"{saida_registrado.iloc[0][['recomendacao', 'valor_sugerido', 'status_parecer']].to_dict()}")

w = WorkspaceClient()
ACAO_ENDPOINT = publicar_endpoint(api_do_workspace(w), P["endpoint_parecer"],
                                  config_endpoint(NOME_MODELO, VERSAO_MODELO, EXPERIMENTO_ID, P["catalogo"], P["schema_ml"]))
print(f"endpoint {P['endpoint_parecer']}: {ACAO_ENDPOINT} com {NOME_MODELO} v{VERSAO_MODELO} (a prontidão é conferida por "
      "scripts/testar_endpoint_parecer.py)")

🔗 View Logged Model at: <workspace>/ml/experiments/1707553725570564/models/m-40451e057cf84ba391a6460603139092
2026/11/21 03:41:14 INFO mlflow.pyfunc: Validating input example against model signature


[imagem: ver os artefatos do run no MLflow]


Unexpected status resolving file endpoints (403, request ID 41ffbb14-c8ab-41ae-af9f-e6f3b9d91263), using the default endpoints.
Successfully registered model 'fe_bar_sinistros_catalog.ml.parecer_sinistro'.


[imagem: ver os artefatos do run no MLflow]


Unexpected status resolving file endpoints (403, request ID 2feb1fcc-7fed-4344-9eb0-d414e64505f0), using the default endpoints.
🔗 Created version '1' of model 'fe_bar_sinistros_catalog.ml.parecer_sinistro': <workspace>/explore/data/models/fe_bar_sinistros_catalog/ml/parecer_sinistro/version/1
Unexpected status resolving file endpoints (403, request ID 57ddbd99-b515-4d1c-abc4-a267215ab680), using the default endpoints.


[imagem: ver os artefatos do run no MLflow]


registrado: fe_bar_sinistros_catalog.ml.parecer_sinistro versão 1 com o prompt v2; teste: {'recomendacao': 'investigar', 'valor_sugerido': 0.0, 'status_parecer': 'ok'}
endpoint sinistros-clima-parecer-dev: criado com fe_bar_sinistros_catalog.ml.parecer_sinistro v1 (a prontidão é conferida por scripts/testar_endpoint_parecer.py)


[imagem: ver os artefatos do run no MLflow]


In [ ]:
resultado = {"dataset": NOME_DATASET, "casos": len(registros), "prompt": NOME_PROMPT,
             "versoes_prompt": VERSOES_PROMPT, "prompt_producao": int(PRODUCAO['versao']), "promovido": PROMOVER,
             "modelo": NOME_MODELO, "versao_modelo": VERSAO_MODELO, "endpoint": P["endpoint_parecer"],
             "acao_endpoint": ACAO_ENDPOINT, "runs": RUNS, "run_registro": RUN_REGISTRO,
             "metricas": {c: {k: round(v, 4) for k, v in m.items()} for c, m in RESULTADOS.items()}}
print(json.dumps(resultado, ensure_ascii=False, indent=2))
dbutils.notebook.exit(json.dumps(resultado, ensure_ascii=False))  # noqa: F821

{"dataset": "fe_bar_sinistros_catalog.ml.avaliacao_parecer_dataset", "casos": 143, "prompt": "fe_bar_sinistros_catalog.ml.prompt_parecer", "versoes_prompt": {"v1": 1, "v2": 2}, "prompt_producao": 2, "promovido": true, "modelo": "fe_bar_sinistros_catalog.ml.parecer_sinistro", "versao_modelo": "1", "endpoint": "sinistros-clima-parecer-dev", "acao_endpoint": "criado", "runs": {"v1": "5ccfd16aba64499892c4a9f51173a2fa", "v2": "19cc0327ff4a444b84d73f037fb22828"}, "run_registro": "1e2b16de8ab342d18095d1e4492d2840", "metricas": {"v1": {"recomendacao_correta/mean": 0.6643, "json_valido/mean": 0.986, "cita_clausula/mean": 0.951, "padrao_identificado/mean": 0.8951, "diretrizes_parecer/mean": 0.6853, "correctness/mean": 0.6643, "valor_dentro_tolerancia_pagamentos": 0.5833, "casos_pagamento_esperado": 84.0}, "v2": {"json_valido/mean": 1.0, "recomendacao_correta/mean": 0.951, "cita_clausula/mean": 0.951, "padrao_identificado/mean": 0.993, "diretrizes_parecer/mean": 0.9231, "correctness/mean": 0.9301